# Analisis Data Bersih (Cleaned Data Analysis & Verification)

Notebook ini memuat analisis data yang sudah digabungkan dan dibersihkan (`data/processed/merged_recipes.csv`) untuk memastikan integritas dan kualitas data sebelum diumpankan ke sistem rekomendasi.

## 1. Setup & Load Data

Kita baca file `merged_recipes.csv` menggunakan `pandas`.

In [1]:
import pandas as pd
import numpy as np
import os

dataset_path = r'../data/processed/merged_recipes.csv'
if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f'Dataset berhasil dimuat! Dimensi data: {df.shape}')
else:
    print('Error: File merged_recipes.csv tidak ditemukan. Jalankan pipeline data_processing.py terlebih dahulu!')

Dataset berhasil dimuat! Dimensi data: (15593, 11)


### Verifikasi Data Kosong (Null/NaN Verification)

Mari kita lakukan verifikasi ketat untuk memastikan tidak ada lagi baris data yang kosong atau rusak setelah pembersihan.

In [2]:
null_counts = df.isnull().sum()
print('Jumlah nilai kosong per kolom:')
print(null_counts)

# Pengecekan asersi untuk memastikan nilai kosong di semua kolom bernilai 0
assert null_counts.sum() == 0, f'Terdeteksi ada {null_counts.sum()} nilai kosong di dataset bersih!'
print('\nVerifikasi Berhasil: Dataset bersih dari data kosong/NaN!')

Jumlah nilai kosong per kolom:
Title                  0
Ingredients            0
Steps                  0
Loves                  0
category               0
Title_Display          0
Ingredients_Display    0
Steps_Display          0
cooking_time           0
spice_level            0
diet_type              0
dtype: int64

Verifikasi Berhasil: Dataset bersih dari data kosong/NaN!


## 2. Pemeriksaan Struktur Kolom dan Data Kosong

Mari kita tampilkan info dataset, tipe data, dan periksa apakah ada nilai kosong (NaN).

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15593 entries, 0 to 15592
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Title                15593 non-null  str  
 1   Ingredients          15593 non-null  str  
 2   Steps                15593 non-null  str  
 3   Loves                15593 non-null  int64
 4   category             15593 non-null  str  
 5   Title_Display        15593 non-null  str  
 6   Ingredients_Display  15593 non-null  str  
 7   Steps_Display        15593 non-null  str  
 8   cooking_time         15593 non-null  int64
 9   spice_level          15593 non-null  str  
 10  diet_type            15593 non-null  str  
dtypes: int64(2), str(9)
memory usage: 21.6 MB


In [4]:
print('Jumlah nilai kosong per kolom:')
df.isnull().sum()

Jumlah nilai kosong per kolom:


Title                  0
Ingredients            0
Steps                  0
Loves                  0
category               0
Title_Display          0
Ingredients_Display    0
Steps_Display          0
cooking_time           0
spice_level            0
diet_type              0
dtype: int64

## 3. Analisis Distribusi Kategori Masakan

Memastikan pembagian kategori masakan berdasarkan protein seimbang.

In [5]:
category_counts = df['category'].value_counts()
print(category_counts)

category
Udang      1994
Tempe      1985
Tahu       1984
Telur      1972
Sapi       1942
Ikan       1932
Ayam       1901
Kambing    1883
Name: count, dtype: int64


## 4. Analisis Klasifikasi Fitur Baru (Diet & Kepedasan)

Memeriksa statistik untuk fitur `diet_type` dan `spice_level`.

In [6]:
print('--- Distribusi Tipe Diet ---')
print(df['diet_type'].value_counts())
print('\n--- Distribusi Tingkat Kepedasan ---')
print(df['spice_level'].value_counts())

--- Distribusi Tipe Diet ---
diet_type
Non-Vegetarian    13426
Vegetarian         2167
Name: count, dtype: int64

--- Distribusi Tingkat Kepedasan ---
spice_level
Pedas          6890
Sedang         6402
Tidak Pedas    2301
Name: count, dtype: int64


## 5. Analisis Fitur Waktu Memasak (`cooking_time`)

Mari kita periksa statistik waktu memasak yang telah diekstrak.

In [7]:
print('Statistik Waktu Memasak (dalam menit):')
df['cooking_time'].describe()

Statistik Waktu Memasak (dalam menit):


count    15593.000000
mean        23.812801
std         43.022506
min          0.000000
25%         20.000000
50%         20.000000
75%         20.000000
max       2100.000000
Name: cooking_time, dtype: float64

## 6. Analisis Resep Terpopuler (`Loves`)

Menampilkan resep masakan terpopuler yang disukai oleh komunitas.

In [8]:
print('Statistik jumlah Loves:')
print(df['Loves'].describe())
print('\nTop 5 Resep Terpopuler:')
df[['Title_Display', 'category', 'diet_type', 'Loves']].sort_values(by='Loves', ascending=False).head(5)

Statistik jumlah Loves:
count    15593.000000
mean        11.779901
std         21.549629
min          0.000000
25%          3.000000
50%          6.000000
75%         11.000000
max        939.000000
Name: Loves, dtype: float64

Top 5 Resep Terpopuler:


,Title_Display,category,diet_type,Loves
7447,Bakso Sapi (Pakai Blender),Sapi,Non-Vegetarian,939
1988,Chilli Tuna Puff Kilat Super Yummy 😙,Ikan,Non-Vegetarian,516
8517,Perkedel Tahu Simple,Tahu,Non-Vegetarian,481
13319,Orek tempe basah bumbu ulek,Tempe,Vegetarian,452
7187,Sop Iga Sapi Enaaak bangeet👌👍,Sapi,Non-Vegetarian,375


## 7. Pemeriksaan Integritas Teks (Sebelum vs Sesudah Pembersihan)

Memastikan teks pembersihan (`Title`, `Ingredients`, `Steps`) sudah bersih untuk TF-IDF, sementara kolom `*_Display` tetap terformat dengan baik.

In [9]:
sample = df.sample(1).iloc[0]
print('=== PERBANDINGAN JUDUL ===')
print('Asli  :', sample['Title_Display'])
print('Bersih:', sample['Title'])
print('\n=== PERBANDINGAN BAHAN ===')
print('Asli  :', sample['Ingredients_Display'][:150], '...')
print('Bersih:', sample['Ingredients'][:150], '...')
print('\n=== PERBANDINGAN LANGKAH ===')
print('Asli  :', sample['Steps_Display'][:150], '...')
print('Bersih:', sample['Steps'][:150], '...')

=== PERBANDINGAN JUDUL ===
Asli  : Shrimp with salted egg
Bersih: shrimp with salted egg

=== PERBANDINGAN BAHAN ===
Asli  : 3 ekor udang ukuran besar (cuci bersih, lumuri garam dan lemon)--1 buah telur ayam--5 sdm tepung bumbu--3 siung bawang putih cincang--1/4 sdt lada put ...
Bersih: 3 ekor udang ukuran besar cuci bersih lumuri garam lemon 1 telur ayam 5 tepung 3 bawang putih cincang 1 4 lada putih bubuk 1 minyak wijen 2 cabe merah ...

=== PERBANDINGAN LANGKAH ===
Asli  : Celupkan udang dalam telur, lumuri dengan tepung bumbu lalu goreng hingga matang, tiriskan--Tumis bawang putih hingga harum, masukkan kuning telur yg  ...
Bersih: celupkan udang dalam telur lumuri tepung matang tiriskan tumis bawang putih harum masukkan kuning telur telah dihancurkan garpu susu mintak wijen lada ...


## Kesimpulan Analisis Data

1. **Integritas Data**: Tidak ada nilai kosong (NaN) pada kolom utama.
2. **Keseimbangan Kelas**: 8 kategori protein terbagi merata (masing-masing sekitar 1.900 resep).
3. **Hasil Preprocessing**: Kolom bersih (`Title`, `Ingredients`, `Steps`) berhasil dikonversi ke format non-tanda-baca dan stopword-free yang siap digunakan untuk kalkulasi TF-IDF dan Cosine Similarity, sementara visual display tetap aman di kolom `Display`.